# Titans Board v2.0 — Colab クイックスタート

CFO / CLO / CEO / 監査役 の AI 取締役会を Colab 上で動かすセットアップです。

**推奨ランタイム**: `T4 GPU`（メニュー → ランタイム → ランタイムのタイプを変更）  
CPUでも動きますが Qwen3-4B の応答が遅くなります（1レスポンス30秒〜）。

---
## Step 0 — ランタイム確認

In [ ]:
import subprocess, sys
!nvidia-smi 2>/dev/null || echo 'CPU ランタイム（GPUなし）'
!free -h | head -2
!df -h / | tail -1

## Step 1 — Ollama インストール & サーバー起動

In [ ]:
%%bash
set -e   # 途中で失敗したらそこで止まる（静かに次へ進まない）

# 1) zstd: Ollama アーカイブの展開に必須（Colab には未導入）
if ! command -v zstd > /dev/null; then
  apt-get update -qq
  apt-get install -y -q zstd
fi
echo "--- zstd: $(zstd --version)"

# 2) Ollama 本体インストール
curl -fsSL https://ollama.com/install.sh | sh

# 3) 成功確認（これが表示されなければ失敗している）
echo "--- ollama: $(ollama --version)"
echo "=== インストール成功 ==="

In [ ]:
import shutil, subprocess, time, urllib.request

# 前のセル（インストール）が成功していないと進めない
assert shutil.which("ollama"), (
    "ollama コマンドが見つかりません。"
    "前のセルを再実行し『=== インストール成功 ===』が出ることを確認してください"
)

# バックグラウンドでサーバー起動
_ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# 起動を最大30秒待つ
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("Ollama サーバー: OK")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama サーバーが30秒以内に起動しませんでした")

## Step 2 — モデル取得

| モデル | サイズ | 用途 |
|--------|--------|------|
| `qwen3:4b` | 約 2.4 GB | 取締役会の推論（必須） |
| `nomic-embed-text` | 約 274 MB | 意味検索（任意・推奨） |

In [ ]:
# 必須: 推論モデル（約2.4GB・数分かかります）
!ollama pull qwen3:4b

In [ ]:
# 任意: 埋め込みモデル（意味検索の精度向上・約274MB）
# 入れない場合はオフラインの hashing embedder が使われます
!ollama pull nomic-embed-text

## Step 3 — リポジトリ取得 & 依存インストール

In [ ]:
# リポジトリクローン（開発ブランチを直接取得 — PRやマージは不要）
# 再実行時はclone済みでも必ず git pull で最新コードに更新する
import os
if not os.path.exists("/content/titans-board"):
    !git clone -b claude/titans-board-v2-design-pb81x7 https://github.com/nori1234/Stock_Tracker-657.git /content/titans-board
%cd /content/titans-board
!git pull
!git log --oneline -1   # ← 実行中のコードのコミットを表示（バグ報告時に便利）

In [ ]:
# Python パッケージインストール（検証済みバージョンに固定）
# crewai==1.14.6 を強制 — Colab に新しい crewai が入っていてもダウングレードされる
!pip install -q -r requirements.txt
import crewai
print("crewai:", crewai.__version__)
assert crewai.__version__ == "1.14.6", "ランタイムを再起動してこのセルを再実行してください"

In [ ]:
# nomic-embed-text を入れた場合は embedder を ollama に切り替える
# （入れていない場合はこのセルをスキップ）
# ※ embedding_dim は変更不要 — OllamaEmbedder がモデルの実次元を自動検出します
import pathlib
cfg = pathlib.Path("config.yaml")
cfg.write_text(cfg.read_text().replace("embedder: hashing", "embedder: ollama"))
print("embedder: ollama に変更しました")
print("→ この後 Step 5（--ingest）を実行すると 768次元で知識が取り込まれます")

## Step 4 — 接続確認

In [ ]:
!python main.py --health-check

`Status: OK` が出れば準備完了です。

---
## Step 5 — 知識ベース取り込み（任意）

In [ ]:
# サンプル知識ファイルを取り込む
!python main.py --ingest ./knowledge

## Step 6 — 長期記憶に方針・禁止事項を登録（任意）

In [ ]:
!python main.py --remember "ギャンブル・アダルト関連事業への参入禁止" --category 禁止事項
!python main.py --remember "3年以内の黒字化を全事業の必須条件とする" --category 経営方針
!python main.py --memories

---
## Step 7 — 取締役会を開催する

CFO → CLO → CEO草稿 → 監査役 → CEO最終 の順で審議が進みます。  
T4 GPU で 1〜3 分、CPU では 5〜15 分程度かかります。

In [ ]:
AGENDA = "新規事業として、AIを活用した医療診断支援サービスを日本市場で展開したい。初期投資5億円、3年でのROI達成が目標。取締役会の判断を仰ぎたい。"

!python main.py "{AGENDA}"

In [ ]:
# 保存されたレポートを確認
import glob, json, pathlib
files = sorted(glob.glob("outputs/meeting_*.json"))
if files:
    latest = json.loads(pathlib.Path(files[-1]).read_text())
    print(f"保存先: {files[-1]}")
    print("\n=== CEO最終判断 ===")
    print(latest.get("ceo_final", "")[:1000])

---
## オプション: Anthropic API を使う場合

Ollama の代わりに Claude を使いたい場合（GPU 不要・応答が速い）。

In [ ]:
import os, pathlib
from google.colab import userdata

# Colab シークレット（左メニュー 🔑）に ANTHROPIC_API_KEY を登録しておく
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# config.yaml を Anthropic に切り替え
cfg = pathlib.Path("config.yaml")
text = cfg.read_text()
text = text.replace("provider: ollama", "provider: anthropic")
text = text.replace("model: qwen3:4b", "model: claude-haiku-4-5-20251001")
cfg.write_text(text)
print("provider: anthropic に切り替えました")

!python main.py --health-check

---
## トラブルシュート

| 症状 | 対処 |
|------|------|
| `requires zstd for extraction` | Step 1 のセルが `apt-get install zstd` を含む最新版か確認して再実行 |
| `ollama serve` で `FileNotFoundError` | インストールが未完了。Step 1 のセルで『=== インストール成功 ===』が出るまで進まない |
| `Status: FAIL` | Step 1 のサーバー起動セルを再実行 |
| `model not found` | `!ollama pull qwen3:4b` を再実行 |
| セッション切れ後に起動しない | Step 1〜2 のセルを再実行（Colab はセッションごとにリセット） |
| GPU メモリ不足 | `qwen3:4b` は約 3GB VRAM。T4（16GB）で余裕あり |
| 応答が遅い | CPU ランタイムの場合。T4 GPU に切り替えると数倍速くなる |